In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("BankDataAnalysis").getOrCreate()

customers = spark.read.csv("bank_customers.csv", header=True, inferSchema=True)
accounts = spark.read.csv("accounts.csv", header=True, inferSchema=True)
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
branches = spark.read.csv("branches.csv", header=True, inferSchema=True)
profiles = spark.read.json("customer_profiles.json", multiLine=True)

# **Section 1 — DataFrame Basics**

In [6]:
# 1
customers.show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [7]:
# 2
accounts.show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1001|          1|   Hyderabad Main|  85000|
|      1002|          2|Bangalore Central| 120000|
|      1003|          3|      Mumbai West|  45000|
|      1004|          4|      Delhi North|  95000|
|      1005|          5|    Chennai South|  60000|
|      1006|          6|   Hyderabad Main| 150000|
|      1007|          7|        Pune East|  30000|
|      1008|          8|      Delhi North|  70000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
|      1011|         11|   Hyderabad Main|  50000|
|      1012|         12|      Delhi North|  40000|
+----------+-----------+-----------------+-------+



In [8]:
# 3
transactions.show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
+------+----------+--------+------+----------+



In [9]:
# 4
branches.show()

+-----------------+------+-------+
|           branch|region|manager|
+-----------------+------+-------+
|   Hyderabad Main| South| Ramesh|
|Bangalore Central| South|  Leena|
|      Mumbai West|  West| Joseph|
|      Delhi North| North|   Sara|
|    Chennai South| South|  Kumar|
|        Pune East|  West|  Anita|
+-----------------+------+-------+



In [10]:
# 5
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [11]:
# 6
transactions.printSchema()


root
 |-- txn_id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- txn_date: date (nullable = true)



In [12]:
# 7
customers.count()

12

In [13]:
# 8
accounts.count()

12

In [14]:
# 9
transactions.count()


10

In [15]:
# 10
transactions.show(5)


+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 5 rows


# **Section 2 — Select, Rename, Filter**

In [16]:
# 11
customers.select("name","city","account_type").show()

+------+---------+------------+
|  name|     city|account_type|
+------+---------+------------+
| Rahul|Hyderabad|     Savings|
| Sneha|Bangalore|     Current|
| Arjun|   Mumbai|     Savings|
| Priya|    Delhi|     Savings|
| Karan|  Chennai|     Current|
| Meera|Hyderabad|     Savings|
|  Amit|     Pune|     Current|
|  Neha|    Delhi|     Savings|
| Divya|Bangalore|     Savings|
|Vikram|   Mumbai|     Current|
|Farhan|Hyderabad|     Savings|
|Simran|    Delhi|     Savings|
+------+---------+------------+



In [17]:
# 12
accounts.select("account_id","balance").show()

+----------+-------+
|account_id|balance|
+----------+-------+
|      1001|  85000|
|      1002| 120000|
|      1003|  45000|
|      1004|  95000|
|      1005|  60000|
|      1006| 150000|
|      1007|  30000|
|      1008|  70000|
|      1009| 110000|
|      1010| 200000|
|      1011|  50000|
|      1012|  40000|
+----------+-------+



In [18]:
# 13
accounts.withColumnRenamed("balance","current_balance").show()

+----------+-----------+-----------------+---------------+
|account_id|customer_id|           branch|current_balance|
+----------+-----------+-----------------+---------------+
|      1001|          1|   Hyderabad Main|          85000|
|      1002|          2|Bangalore Central|         120000|
|      1003|          3|      Mumbai West|          45000|
|      1004|          4|      Delhi North|          95000|
|      1005|          5|    Chennai South|          60000|
|      1006|          6|   Hyderabad Main|         150000|
|      1007|          7|        Pune East|          30000|
|      1008|          8|      Delhi North|          70000|
|      1009|          9|Bangalore Central|         110000|
|      1010|         10|      Mumbai West|         200000|
|      1011|         11|   Hyderabad Main|          50000|
|      1012|         12|      Delhi North|          40000|
+----------+-----------+-----------------+---------------+



In [19]:
# 14
transactions.withColumnRenamed("txn_type","transaction_type").show()


+------+----------+----------------+------+----------+
|txn_id|account_id|transaction_type|amount|  txn_date|
+------+----------+----------------+------+----------+
|     1|      1001|          Credit| 25000|2024-03-01|
|     2|      1002|           Debit| 15000|2024-03-01|
|     3|      1003|          Credit| 10000|2024-03-02|
|     4|      1004|           Debit|  5000|2024-03-02|
|     5|      1005|          Credit| 30000|2024-03-03|
|     6|      1006|           Debit| 20000|2024-03-03|
|     7|      1007|          Credit|  8000|2024-03-04|
|     8|      1008|           Debit| 12000|2024-03-04|
|     9|      1009|          Credit| 40000|2024-03-05|
|    10|      1010|           Debit| 35000|2024-03-05|
+------+----------+----------------+------+----------+



In [20]:
# 15
customers.filter(col("city")=="Hyderabad").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [21]:
# 16
customers.filter(col("age")>30).show()


+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
+-----------+------+---------+---+------------+-----------+



In [22]:
# 17
customers.filter(col("account_type")=="Savings").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [23]:
# 18
accounts.filter(col("balance")>100000).show()


+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1002|          2|Bangalore Central| 120000|
|      1006|          6|   Hyderabad Main| 150000|
|      1009|          9|Bangalore Central| 110000|
|      1010|         10|      Mumbai West| 200000|
+----------+-----------+-----------------+-------+



In [24]:
# 19
transactions.filter(col("txn_type")=="Credit").show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
+------+----------+--------+------+----------+



In [25]:
# 20
transactions.filter(col("amount")>20000).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     5|      1005|  Credit| 30000|2024-03-03|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
+------+----------+--------+------+----------+



# **Section 3 — Sorting and Limit**

In [26]:
# 21
customers.orderBy("age").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
+-----------+------+---------+---+------------+-----------+



In [27]:
# 22
customers.orderBy(col("age").desc()).show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [28]:
# 23
accounts.orderBy(col("balance").desc()).show()

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1010|         10|      Mumbai West| 200000|
|      1006|          6|   Hyderabad Main| 150000|
|      1002|          2|Bangalore Central| 120000|
|      1009|          9|Bangalore Central| 110000|
|      1004|          4|      Delhi North|  95000|
|      1001|          1|   Hyderabad Main|  85000|
|      1008|          8|      Delhi North|  70000|
|      1005|          5|    Chennai South|  60000|
|      1011|         11|   Hyderabad Main|  50000|
|      1003|          3|      Mumbai West|  45000|
|      1012|         12|      Delhi North|  40000|
|      1007|          7|        Pune East|  30000|
+----------+-----------+-----------------+-------+



In [29]:
# 24
accounts.orderBy(col("balance").desc()).show(5)

+----------+-----------+-----------------+-------+
|account_id|customer_id|           branch|balance|
+----------+-----------+-----------------+-------+
|      1010|         10|      Mumbai West| 200000|
|      1006|          6|   Hyderabad Main| 150000|
|      1002|          2|Bangalore Central| 120000|
|      1009|          9|Bangalore Central| 110000|
|      1004|          4|      Delhi North|  95000|
+----------+-----------+-----------------+-------+
only showing top 5 rows


In [30]:
# 25
accounts.orderBy("balance").show(5)

+----------+-----------+--------------+-------+
|account_id|customer_id|        branch|balance|
+----------+-----------+--------------+-------+
|      1007|          7|     Pune East|  30000|
|      1012|         12|   Delhi North|  40000|
|      1003|          3|   Mumbai West|  45000|
|      1011|         11|Hyderabad Main|  50000|
|      1005|          5| Chennai South|  60000|
+----------+-----------+--------------+-------+
only showing top 5 rows


In [31]:
# 26
transactions.orderBy(col("amount").desc()).show()


+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|     5|      1005|  Credit| 30000|2024-03-03|
|     1|      1001|  Credit| 25000|2024-03-01|
|     6|      1006|   Debit| 20000|2024-03-03|
|     2|      1002|   Debit| 15000|2024-03-01|
|     8|      1008|   Debit| 12000|2024-03-04|
|     3|      1003|  Credit| 10000|2024-03-02|
|     7|      1007|  Credit|  8000|2024-03-04|
|     4|      1004|   Debit|  5000|2024-03-02|
+------+----------+--------+------+----------+



In [32]:
# 27
transactions.orderBy(col("amount").desc()).show(3)

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 3 rows


In [33]:
# 28
transactions.orderBy("txn_date").show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
+------+----------+--------+------+----------+



In [34]:
# 29
customers.orderBy("city","age").show()


+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
+-----------+------+---------+---+------------+-----------+



In [35]:
# 30
transactions.orderBy(col("txn_date").desc()).show(5)


+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     5|      1005|  Credit| 30000|2024-03-03|
+------+----------+--------+------+----------+
only showing top 5 rows


# **Section 4 — Aggregations**

In [36]:
# 31
accounts.agg(sum("balance")).show()


+------------+
|sum(balance)|
+------------+
|     1055000|
+------------+



In [37]:
# 32
accounts.agg(avg("balance")).show()


+-----------------+
|     avg(balance)|
+-----------------+
|87916.66666666667|
+-----------------+



In [38]:
# 33
accounts.agg(max("balance")).show()


+------------+
|max(balance)|
+------------+
|      200000|
+------------+



In [39]:
# 34
accounts.agg(min("balance")).show()


+------------+
|min(balance)|
+------------+
|       30000|
+------------+



In [40]:
# 35
customers.groupBy("city").count().show()


+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [41]:
# 36
customers.groupBy("account_type").count().show()

+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|    8|
|     Current|    4|
+------------+-----+



In [42]:
# 37
transactions.groupBy("txn_type").count().show()

+--------+-----+
|txn_type|count|
+--------+-----+
|  Credit|    5|
|   Debit|    5|
+--------+-----+



In [43]:
# 38
transactions.filter(col("txn_type")=="Credit").agg(sum("amount")).show()


+-----------+
|sum(amount)|
+-----------+
|     113000|
+-----------+



In [44]:
# 39
transactions.filter(col("txn_type")=="Debit").agg(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|      87000|
+-----------+



In [45]:
# 40
transactions.agg(avg("amount")).show()

+-----------+
|avg(amount)|
+-----------+
|    20000.0|
+-----------+



# **Section 5 — GroupBy Analytics**

In [46]:
# 41
accounts.groupBy("branch").agg(sum("balance")).show()


+-----------------+------------+
|           branch|sum(balance)|
+-----------------+------------+
|      Delhi North|      205000|
|   Hyderabad Main|      285000|
|        Pune East|       30000|
|      Mumbai West|      245000|
|    Chennai South|       60000|
|Bangalore Central|      230000|
+-----------------+------------+



In [47]:
# 42
accounts.groupBy("branch").agg(avg("balance")).show()


+-----------------+-----------------+
|           branch|     avg(balance)|
+-----------------+-----------------+
|      Delhi North|68333.33333333333|
|   Hyderabad Main|          95000.0|
|        Pune East|          30000.0|
|      Mumbai West|         122500.0|
|    Chennai South|          60000.0|
|Bangalore Central|         115000.0|
+-----------------+-----------------+



In [49]:
# 43
accounts.groupBy("branch").count().show()

+-----------------+-----+
|           branch|count|
+-----------------+-----+
|      Delhi North|    3|
|   Hyderabad Main|    3|
|        Pune East|    1|
|      Mumbai West|    2|
|    Chennai South|    1|
|Bangalore Central|    2|
+-----------------+-----+



In [48]:
# 44
transactions.groupBy("account_id").agg(sum("amount")).show()

+----------+-----------+
|account_id|sum(amount)|
+----------+-----------+
|      1005|      30000|
|      1008|      12000|
|      1010|      35000|
|      1002|      15000|
|      1001|      25000|
|      1006|      20000|
|      1007|       8000|
|      1003|      10000|
|      1004|       5000|
|      1009|      40000|
+----------+-----------+



In [50]:
# 45
transactions.groupBy("account_id").count().show()

+----------+-----+
|account_id|count|
+----------+-----+
|      1005|    1|
|      1008|    1|
|      1010|    1|
|      1002|    1|
|      1001|    1|
|      1006|    1|
|      1007|    1|
|      1003|    1|
|      1004|    1|
|      1009|    1|
+----------+-----+



In [51]:
# 46
transactions.groupBy("txn_type").agg(sum("amount")).show()

+--------+-----------+
|txn_type|sum(amount)|
+--------+-----------+
|  Credit|     113000|
|   Debit|      87000|
+--------+-----------+



In [52]:
# 47
transactions.groupBy("txn_date").count().show()


+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-05|    2|
|2024-03-04|    2|
|2024-03-02|    2|
|2024-03-01|    2|
|2024-03-03|    2|
+----------+-----+



In [53]:
# 48
customers.groupBy("city").agg(avg("age")).show()

+---------+------------------+
|     city|          avg(age)|
+---------+------------------+
|Bangalore|              30.0|
|  Chennai|              30.0|
|   Mumbai|              34.5|
|     Pune|              38.0|
|    Delhi|28.666666666666668|
|Hyderabad|31.333333333333332|
+---------+------------------+



In [54]:
# 49
customers.join(accounts,"customer_id").groupBy("account_type").agg(sum("balance")).show()



+------------+------------+
|account_type|sum(balance)|
+------------+------------+
|     Savings|      645000|
|     Current|      410000|
+------------+------------+



In [55]:
# 50
customers.groupBy("city","account_type").count().show()

+---------+------------+-----+
|     city|account_type|count|
+---------+------------+-----+
|   Mumbai|     Savings|    1|
|Hyderabad|     Savings|    3|
|Bangalore|     Savings|    1|
|Bangalore|     Current|    1|
|   Mumbai|     Current|    1|
|    Delhi|     Savings|    3|
|     Pune|     Current|    1|
|  Chennai|     Current|    1|
+---------+------------+-----+



## **Section 6 — Joins**

In [56]:
# 51
customers.join(accounts,"customer_id").show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|      1001|   Hyderabad Main|  85000|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      1003|      Mumbai West|  45000|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|
|          8|  Neha|    Delhi|

In [57]:
# 52
customers.join(accounts,"customer_id").select("name","city","branch","balance").show()


+------+---------+-----------------+-------+
|  name|     city|           branch|balance|
+------+---------+-----------------+-------+
| Rahul|Hyderabad|   Hyderabad Main|  85000|
| Sneha|Bangalore|Bangalore Central| 120000|
| Arjun|   Mumbai|      Mumbai West|  45000|
| Priya|    Delhi|      Delhi North|  95000|
| Karan|  Chennai|    Chennai South|  60000|
| Meera|Hyderabad|   Hyderabad Main| 150000|
|  Amit|     Pune|        Pune East|  30000|
|  Neha|    Delhi|      Delhi North|  70000|
| Divya|Bangalore|Bangalore Central| 110000|
|Vikram|   Mumbai|      Mumbai West| 200000|
|Farhan|Hyderabad|   Hyderabad Main|  50000|
|Simran|    Delhi|      Delhi North|  40000|
+------+---------+-----------------+-------+



In [58]:
# 53
accounts.join(transactions,"account_id").show()

+----------+-----------+-----------------+-------+------+--------+------+----------+
|account_id|customer_id|           branch|balance|txn_id|txn_type|amount|  txn_date|
+----------+-----------+-----------------+-------+------+--------+------+----------+
|      1001|          1|   Hyderabad Main|  85000|     1|  Credit| 25000|2024-03-01|
|      1002|          2|Bangalore Central| 120000|     2|   Debit| 15000|2024-03-01|
|      1003|          3|      Mumbai West|  45000|     3|  Credit| 10000|2024-03-02|
|      1004|          4|      Delhi North|  95000|     4|   Debit|  5000|2024-03-02|
|      1005|          5|    Chennai South|  60000|     5|  Credit| 30000|2024-03-03|
|      1006|          6|   Hyderabad Main| 150000|     6|   Debit| 20000|2024-03-03|
|      1007|          7|        Pune East|  30000|     7|  Credit|  8000|2024-03-04|
|      1008|          8|      Delhi North|  70000|     8|   Debit| 12000|2024-03-04|
|      1009|          9|Bangalore Central| 110000|     9|  Credit

In [59]:
# 54
accounts.join(transactions,"account_id").select("account_id","txn_type","amount","balance").show()

+----------+--------+------+-------+
|account_id|txn_type|amount|balance|
+----------+--------+------+-------+
|      1001|  Credit| 25000|  85000|
|      1002|   Debit| 15000| 120000|
|      1003|  Credit| 10000|  45000|
|      1004|   Debit|  5000|  95000|
|      1005|  Credit| 30000|  60000|
|      1006|   Debit| 20000| 150000|
|      1007|  Credit|  8000|  30000|
|      1008|   Debit| 12000|  70000|
|      1009|  Credit| 40000| 110000|
|      1010|   Debit| 35000| 200000|
+----------+--------+------+-------+



In [60]:
# 55
accounts.join(branches,"branch").show()

+-----------------+----------+-----------+-------+------+-------+
|           branch|account_id|customer_id|balance|region|manager|
+-----------------+----------+-----------+-------+------+-------+
|   Hyderabad Main|      1001|          1|  85000| South| Ramesh|
|Bangalore Central|      1002|          2| 120000| South|  Leena|
|      Mumbai West|      1003|          3|  45000|  West| Joseph|
|      Delhi North|      1004|          4|  95000| North|   Sara|
|    Chennai South|      1005|          5|  60000| South|  Kumar|
|   Hyderabad Main|      1006|          6| 150000| South| Ramesh|
|        Pune East|      1007|          7|  30000|  West|  Anita|
|      Delhi North|      1008|          8|  70000| North|   Sara|
|Bangalore Central|      1009|          9| 110000| South|  Leena|
|      Mumbai West|      1010|         10| 200000|  West| Joseph|
|   Hyderabad Main|      1011|         11|  50000| South| Ramesh|
|      Delhi North|      1012|         12|  40000| North|   Sara|
+---------

In [61]:
# 56
accounts.join(branches,"branch").select("branch","region","manager","balance").show()


+-----------------+------+-------+-------+
|           branch|region|manager|balance|
+-----------------+------+-------+-------+
|   Hyderabad Main| South| Ramesh|  85000|
|Bangalore Central| South|  Leena| 120000|
|      Mumbai West|  West| Joseph|  45000|
|      Delhi North| North|   Sara|  95000|
|    Chennai South| South|  Kumar|  60000|
|   Hyderabad Main| South| Ramesh| 150000|
|        Pune East|  West|  Anita|  30000|
|      Delhi North| North|   Sara|  70000|
|Bangalore Central| South|  Leena| 110000|
|      Mumbai West|  West| Joseph| 200000|
|   Hyderabad Main| South| Ramesh|  50000|
|      Delhi North| North|   Sara|  40000|
+-----------------+------+-------+-------+



In [62]:
# 57
customers.join(accounts,"customer_id").join(transactions,"account_id").show()

+----------+-----------+------+---------+---+------------+-----------+-----------------+-------+------+--------+------+----------+
|account_id|customer_id|  name|     city|age|account_type|signup_date|           branch|balance|txn_id|txn_type|amount|  txn_date|
+----------+-----------+------+---------+---+------------+-----------+-----------------+-------+------+--------+------+----------+
|      1001|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|   Hyderabad Main|  85000|     1|  Credit| 25000|2024-03-01|
|      1002|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|Bangalore Central| 120000|     2|   Debit| 15000|2024-03-01|
|      1003|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|      Mumbai West|  45000|     3|  Credit| 10000|2024-03-02|
|      1004|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      Delhi North|  95000|     4|   Debit|  5000|2024-03-02|
|      1005|          5| Karan|  Chennai| 30|     Current| 2023-05-11|    Chennai S

In [63]:
# 58
customers.join(accounts,"customer_id").join(transactions,"account_id")\
.select("name","city","txn_type","amount","txn_date").show()


+------+---------+--------+------+----------+
|  name|     city|txn_type|amount|  txn_date|
+------+---------+--------+------+----------+
| Rahul|Hyderabad|  Credit| 25000|2024-03-01|
| Sneha|Bangalore|   Debit| 15000|2024-03-01|
| Arjun|   Mumbai|  Credit| 10000|2024-03-02|
| Priya|    Delhi|   Debit|  5000|2024-03-02|
| Karan|  Chennai|  Credit| 30000|2024-03-03|
| Meera|Hyderabad|   Debit| 20000|2024-03-03|
|  Amit|     Pune|  Credit|  8000|2024-03-04|
|  Neha|    Delhi|   Debit| 12000|2024-03-04|
| Divya|Bangalore|  Credit| 40000|2024-03-05|
|Vikram|   Mumbai|   Debit| 35000|2024-03-05|
+------+---------+--------+------+----------+



In [64]:
# 59
customers.join(accounts,"customer_id").join(branches,"branch").join(transactions,"account_id").show()

+----------+-----------------+-----------+------+---------+---+------------+-----------+-------+------+-------+------+--------+------+----------+
|account_id|           branch|customer_id|  name|     city|age|account_type|signup_date|balance|region|manager|txn_id|txn_type|amount|  txn_date|
+----------+-----------------+-----------+------+---------+---+------------+-----------+-------+------+-------+------+--------+------+----------+
|      1001|   Hyderabad Main|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  85000| South| Ramesh|     1|  Credit| 25000|2024-03-01|
|      1002|Bangalore Central|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| 120000| South|  Leena|     2|   Debit| 15000|2024-03-01|
|      1003|      Mumbai West|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|  45000|  West| Joseph|     3|  Credit| 10000|2024-03-02|
|      1004|      Delhi North|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|  95000| North|   Sara|     4|   Deb

In [65]:
# 60
customers.join(accounts,"customer_id").join(transactions,"account_id")\
.groupBy("name").agg(sum("amount")).show()

+------+-----------+
|  name|sum(amount)|
+------+-----------+
| Divya|      40000|
| Meera|      20000|
| Sneha|      15000|
| Priya|       5000|
|Vikram|      35000|
| Rahul|      25000|
| Arjun|      10000|
|  Amit|       8000|
|  Neha|      12000|
| Karan|      30000|
+------+-----------+



# **Section 7 — Updating Data with withColumn**

In [66]:
# 61
accounts.withColumn("balance_in_lakhs", col("balance")/100000).show()


+----------+-----------+-----------------+-------+----------------+
|account_id|customer_id|           branch|balance|balance_in_lakhs|
+----------+-----------+-----------------+-------+----------------+
|      1001|          1|   Hyderabad Main|  85000|            0.85|
|      1002|          2|Bangalore Central| 120000|             1.2|
|      1003|          3|      Mumbai West|  45000|            0.45|
|      1004|          4|      Delhi North|  95000|            0.95|
|      1005|          5|    Chennai South|  60000|             0.6|
|      1006|          6|   Hyderabad Main| 150000|             1.5|
|      1007|          7|        Pune East|  30000|             0.3|
|      1008|          8|      Delhi North|  70000|             0.7|
|      1009|          9|Bangalore Central| 110000|             1.1|
|      1010|         10|      Mumbai West| 200000|             2.0|
|      1011|         11|   Hyderabad Main|  50000|             0.5|
|      1012|         12|      Delhi North|  4000

In [67]:
# 62
accounts.withColumn("bank_name", lit("BotCampus Bank")).show()

+----------+-----------+-----------------+-------+--------------+
|account_id|customer_id|           branch|balance|     bank_name|
+----------+-----------+-----------------+-------+--------------+
|      1001|          1|   Hyderabad Main|  85000|BotCampus Bank|
|      1002|          2|Bangalore Central| 120000|BotCampus Bank|
|      1003|          3|      Mumbai West|  45000|BotCampus Bank|
|      1004|          4|      Delhi North|  95000|BotCampus Bank|
|      1005|          5|    Chennai South|  60000|BotCampus Bank|
|      1006|          6|   Hyderabad Main| 150000|BotCampus Bank|
|      1007|          7|        Pune East|  30000|BotCampus Bank|
|      1008|          8|      Delhi North|  70000|BotCampus Bank|
|      1009|          9|Bangalore Central| 110000|BotCampus Bank|
|      1010|         10|      Mumbai West| 200000|BotCampus Bank|
|      1011|         11|   Hyderabad Main|  50000|BotCampus Bank|
|      1012|         12|      Delhi North|  40000|BotCampus Bank|
+---------

In [68]:
# 63
accounts.withColumn("annual_service_fee", col("balance")*0.01).show()

+----------+-----------+-----------------+-------+------------------+
|account_id|customer_id|           branch|balance|annual_service_fee|
+----------+-----------+-----------------+-------+------------------+
|      1001|          1|   Hyderabad Main|  85000|             850.0|
|      1002|          2|Bangalore Central| 120000|            1200.0|
|      1003|          3|      Mumbai West|  45000|             450.0|
|      1004|          4|      Delhi North|  95000|             950.0|
|      1005|          5|    Chennai South|  60000|             600.0|
|      1006|          6|   Hyderabad Main| 150000|            1500.0|
|      1007|          7|        Pune East|  30000|             300.0|
|      1008|          8|      Delhi North|  70000|             700.0|
|      1009|          9|Bangalore Central| 110000|            1100.0|
|      1010|         10|      Mumbai West| 200000|            2000.0|
|      1011|         11|   Hyderabad Main|  50000|             500.0|
|      1012|        

In [69]:
# 64
accounts.withColumn("annual_service_fee", col("balance")*0.01)\
.withColumn("net_balance", col("balance")-col("annual_service_fee")).show()

+----------+-----------+-----------------+-------+------------------+-----------+
|account_id|customer_id|           branch|balance|annual_service_fee|net_balance|
+----------+-----------+-----------------+-------+------------------+-----------+
|      1001|          1|   Hyderabad Main|  85000|             850.0|    84150.0|
|      1002|          2|Bangalore Central| 120000|            1200.0|   118800.0|
|      1003|          3|      Mumbai West|  45000|             450.0|    44550.0|
|      1004|          4|      Delhi North|  95000|             950.0|    94050.0|
|      1005|          5|    Chennai South|  60000|             600.0|    59400.0|
|      1006|          6|   Hyderabad Main| 150000|            1500.0|   148500.0|
|      1007|          7|        Pune East|  30000|             300.0|    29700.0|
|      1008|          8|      Delhi North|  70000|             700.0|    69300.0|
|      1009|          9|Bangalore Central| 110000|            1100.0|   108900.0|
|      1010|    

In [70]:
# 65
accounts.withColumn("is_high_balance", col("balance")>100000).show()

+----------+-----------+-----------------+-------+---------------+
|account_id|customer_id|           branch|balance|is_high_balance|
+----------+-----------+-----------------+-------+---------------+
|      1001|          1|   Hyderabad Main|  85000|          false|
|      1002|          2|Bangalore Central| 120000|           true|
|      1003|          3|      Mumbai West|  45000|          false|
|      1004|          4|      Delhi North|  95000|          false|
|      1005|          5|    Chennai South|  60000|          false|
|      1006|          6|   Hyderabad Main| 150000|           true|
|      1007|          7|        Pune East|  30000|          false|
|      1008|          8|      Delhi North|  70000|          false|
|      1009|          9|Bangalore Central| 110000|           true|
|      1010|         10|      Mumbai West| 200000|           true|
|      1011|         11|   Hyderabad Main|  50000|          false|
|      1012|         12|      Delhi North|  40000|          fa

In [71]:
# 66
transactions.withColumn("txn_amount_in_k", col("amount")/1000).show()

+------+----------+--------+------+----------+---------------+
|txn_id|account_id|txn_type|amount|  txn_date|txn_amount_in_k|
+------+----------+--------+------+----------+---------------+
|     1|      1001|  Credit| 25000|2024-03-01|           25.0|
|     2|      1002|   Debit| 15000|2024-03-01|           15.0|
|     3|      1003|  Credit| 10000|2024-03-02|           10.0|
|     4|      1004|   Debit|  5000|2024-03-02|            5.0|
|     5|      1005|  Credit| 30000|2024-03-03|           30.0|
|     6|      1006|   Debit| 20000|2024-03-03|           20.0|
|     7|      1007|  Credit|  8000|2024-03-04|            8.0|
|     8|      1008|   Debit| 12000|2024-03-04|           12.0|
|     9|      1009|  Credit| 40000|2024-03-05|           40.0|
|    10|      1010|   Debit| 35000|2024-03-05|           35.0|
+------+----------+--------+------+----------+---------------+



In [72]:
# 67
customers.withColumn("country", lit("India")).show()

+-----------+------+---------+---+------------+-----------+-------+
|customer_id|  name|     city|age|account_type|signup_date|country|
+-----------+------+---------+---+------------+-----------+-------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|  India|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|  India|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|  India|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|  India|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|  India|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|  India|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|  India|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|  India|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|  India|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|  India|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|  India|
|         12|Simran|    Delhi| 25|     Savings| 

In [73]:
# 68
customers.withColumn("customer_label", concat(col("name"),lit(" - "),col("city"))).show()


+-----------+------+---------+---+------------+-----------+------------------+
|customer_id|  name|     city|age|account_type|signup_date|    customer_label|
+-----------+------+---------+---+------------+-----------+------------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10| Rahul - Hyderabad|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12| Sneha - Bangalore|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Arjun - Mumbai|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|     Priya - Delhi|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|   Karan - Chennai|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10| Meera - Hyderabad|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|       Amit - Pune|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      Neha - Delhi|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15| Divya - Bangalore|
|         10|Vikram|   Mumbai| 42|     Current| 2023

In [74]:
# 69
accounts.join(branches,"branch")\
.withColumn("branch_label", concat(col("branch"),lit(" - "),col("region"))).show()


+-----------------+----------+-----------+-------+------+-------+--------------------+
|           branch|account_id|customer_id|balance|region|manager|        branch_label|
+-----------------+----------+-----------+-------+------+-------+--------------------+
|   Hyderabad Main|      1001|          1|  85000| South| Ramesh|Hyderabad Main - ...|
|Bangalore Central|      1002|          2| 120000| South|  Leena|Bangalore Central...|
|      Mumbai West|      1003|          3|  45000|  West| Joseph|  Mumbai West - West|
|      Delhi North|      1004|          4|  95000| North|   Sara| Delhi North - North|
|    Chennai South|      1005|          5|  60000| South|  Kumar|Chennai South - S...|
|   Hyderabad Main|      1006|          6| 150000| South| Ramesh|Hyderabad Main - ...|
|        Pune East|      1007|          7|  30000|  West|  Anita|    Pune East - West|
|      Delhi North|      1008|          8|  70000| North|   Sara| Delhi North - North|
|Bangalore Central|      1009|          9| 

In [75]:
# 70
transactions.withColumn("risk_flag", col("amount")>40000).show()


+------+----------+--------+------+----------+---------+
|txn_id|account_id|txn_type|amount|  txn_date|risk_flag|
+------+----------+--------+------+----------+---------+
|     1|      1001|  Credit| 25000|2024-03-01|    false|
|     2|      1002|   Debit| 15000|2024-03-01|    false|
|     3|      1003|  Credit| 10000|2024-03-02|    false|
|     4|      1004|   Debit|  5000|2024-03-02|    false|
|     5|      1005|  Credit| 30000|2024-03-03|    false|
|     6|      1006|   Debit| 20000|2024-03-03|    false|
|     7|      1007|  Credit|  8000|2024-03-04|    false|
|     8|      1008|   Debit| 12000|2024-03-04|    false|
|     9|      1009|  Credit| 40000|2024-03-05|    false|
|    10|      1010|   Debit| 35000|2024-03-05|    false|
+------+----------+--------+------+----------+---------+



# **Section 8 — when / otherwise**

In [77]:
# 71
accounts.withColumn("category",
    when(col("balance")>=100000,"High")
    .when(col("balance")>=50000,"Medium")
    .otherwise("Low")).show()

+----------+-----------+-----------------+-------+--------+
|account_id|customer_id|           branch|balance|category|
+----------+-----------+-----------------+-------+--------+
|      1001|          1|   Hyderabad Main|  85000|  Medium|
|      1002|          2|Bangalore Central| 120000|    High|
|      1003|          3|      Mumbai West|  45000|     Low|
|      1004|          4|      Delhi North|  95000|  Medium|
|      1005|          5|    Chennai South|  60000|  Medium|
|      1006|          6|   Hyderabad Main| 150000|    High|
|      1007|          7|        Pune East|  30000|     Low|
|      1008|          8|      Delhi North|  70000|  Medium|
|      1009|          9|Bangalore Central| 110000|    High|
|      1010|         10|      Mumbai West| 200000|    High|
|      1011|         11|   Hyderabad Main|  50000|  Medium|
|      1012|         12|      Delhi North|  40000|     Low|
+----------+-----------+-----------------+-------+--------+



In [76]:
# 72
customers.withColumn("age_group",
    when(col("age")<30,"Young")
    .when(col("age")<40,"Adult")
    .otherwise("Senior")).show()


+-----------+------+---------+---+------------+-----------+---------+
|customer_id|  name|     city|age|account_type|signup_date|age_group|
+-----------+------+---------+---+------------+-----------+---------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|    Young|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|    Adult|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Young|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|    Adult|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|    Adult|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|    Adult|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|    Adult|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|    Young|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|    Young|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|   Senior|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|    Adult|
|         12|Simran|

In [78]:
# 73
transactions.withColumn("txn_category",
    when(col("amount")>=30000,"Large")
    .when(col("amount")>=10000,"Medium")
    .otherwise("Small")).show()


+------+----------+--------+------+----------+------------+
|txn_id|account_id|txn_type|amount|  txn_date|txn_category|
+------+----------+--------+------+----------+------------+
|     1|      1001|  Credit| 25000|2024-03-01|      Medium|
|     2|      1002|   Debit| 15000|2024-03-01|      Medium|
|     3|      1003|  Credit| 10000|2024-03-02|      Medium|
|     4|      1004|   Debit|  5000|2024-03-02|       Small|
|     5|      1005|  Credit| 30000|2024-03-03|       Large|
|     6|      1006|   Debit| 20000|2024-03-03|      Medium|
|     7|      1007|  Credit|  8000|2024-03-04|       Small|
|     8|      1008|   Debit| 12000|2024-03-04|      Medium|
|     9|      1009|  Credit| 40000|2024-03-05|       Large|
|    10|      1010|   Debit| 35000|2024-03-05|       Large|
+------+----------+--------+------+----------+------------+



In [79]:
# 74
branches.withColumn("priority",
    when(col("region")=="South","High Priority")
    .when(col("region")=="North","Medium Priority")
    .otherwise("Watch")).show()


+-----------------+------+-------+---------------+
|           branch|region|manager|       priority|
+-----------------+------+-------+---------------+
|   Hyderabad Main| South| Ramesh|  High Priority|
|Bangalore Central| South|  Leena|  High Priority|
|      Mumbai West|  West| Joseph|          Watch|
|      Delhi North| North|   Sara|Medium Priority|
|    Chennai South| South|  Kumar|  High Priority|
|        Pune East|  West|  Anita|          Watch|
+-----------------+------+-------+---------------+



In [80]:
# 75
customers.withColumn("type_label",
    when(col("account_type")=="Savings","Retail")
    .otherwise("Business")).show()

+-----------+------+---------+---+------------+-----------+----------+
|customer_id|  name|     city|age|account_type|signup_date|type_label|
+-----------+------+---------+---+------------+-----------+----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|    Retail|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|  Business|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    Retail|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|    Retail|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|  Business|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|    Retail|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|  Business|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|    Retail|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|    Retail|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|  Business|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|    Retail|
|     

# **Section 9 — Date Functions**

In [81]:
# 76
customers.withColumn("signup_date", to_date("signup_date")).show()


+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [82]:
# 77
customers.withColumn("year", year("signup_date")).show()


+-----------+------+---------+---+------------+-----------+----+
|customer_id|  name|     city|age|account_type|signup_date|year|
+-----------+------+---------+---+------------+-----------+----+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|2023|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|2023|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|2023|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|2023|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|2023|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|2023|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|2023|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|2023|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|2023|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|2023|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|2023|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|2023|
+-----------+------+-----

In [83]:
# 78
customers.withColumn("month", month("signup_date")).show()

+-----------+------+---------+---+------------+-----------+-----+
|customer_id|  name|     city|age|account_type|signup_date|month|
+-----------+------+---------+---+------------+-----------+-----+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|    1|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|    2|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|    3|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|    4|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|    5|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|    6|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|    6|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|    7|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|    7|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|    8|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|    8|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|    8|
+---------

In [84]:
# 79
transactions.withColumn("txn_date", to_date("txn_date")).show()

+------+----------+--------+------+----------+
|txn_id|account_id|txn_type|amount|  txn_date|
+------+----------+--------+------+----------+
|     1|      1001|  Credit| 25000|2024-03-01|
|     2|      1002|   Debit| 15000|2024-03-01|
|     3|      1003|  Credit| 10000|2024-03-02|
|     4|      1004|   Debit|  5000|2024-03-02|
|     5|      1005|  Credit| 30000|2024-03-03|
|     6|      1006|   Debit| 20000|2024-03-03|
|     7|      1007|  Credit|  8000|2024-03-04|
|     8|      1008|   Debit| 12000|2024-03-04|
|     9|      1009|  Credit| 40000|2024-03-05|
|    10|      1010|   Debit| 35000|2024-03-05|
+------+----------+--------+------+----------+



In [85]:
# 80
transactions.withColumn("month", month("txn_date")).show()


+------+----------+--------+------+----------+-----+
|txn_id|account_id|txn_type|amount|  txn_date|month|
+------+----------+--------+------+----------+-----+
|     1|      1001|  Credit| 25000|2024-03-01|    3|
|     2|      1002|   Debit| 15000|2024-03-01|    3|
|     3|      1003|  Credit| 10000|2024-03-02|    3|
|     4|      1004|   Debit|  5000|2024-03-02|    3|
|     5|      1005|  Credit| 30000|2024-03-03|    3|
|     6|      1006|   Debit| 20000|2024-03-03|    3|
|     7|      1007|  Credit|  8000|2024-03-04|    3|
|     8|      1008|   Debit| 12000|2024-03-04|    3|
|     9|      1009|  Credit| 40000|2024-03-05|    3|
|    10|      1010|   Debit| 35000|2024-03-05|    3|
+------+----------+--------+------+----------+-----+



In [86]:
# 81
transactions.groupBy("txn_date").count().show()


+----------+-----+
|  txn_date|count|
+----------+-----+
|2024-03-05|    2|
|2024-03-04|    2|
|2024-03-02|    2|
|2024-03-01|    2|
|2024-03-03|    2|
+----------+-----+



In [87]:
# 82
transactions.withColumn("month", month("txn_date")).groupBy("month").count().show()


+-----+-----+
|month|count|
+-----+-----+
|    3|   10|
+-----+-----+



In [88]:
# 83
customers.filter(col("signup_date")>"2023-06-01").show()

+-----------+------+---------+---+------------+-----------+
|customer_id|  name|     city|age|account_type|signup_date|
+-----------+------+---------+---+------------+-----------+
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|
|         11|Farhan|Hyderabad| 34|     Savings| 2023-08-10|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|
+-----------+------+---------+---+------------+-----------+



In [89]:
# 84
customers.withColumn("days_since_signup", datediff(current_date(), col("signup_date"))).show()


+-----------+------+---------+---+------------+-----------+-----------------+
|customer_id|  name|     city|age|account_type|signup_date|days_since_signup|
+-----------+------+---------+---+------------+-----------+-----------------+
|          1| Rahul|Hyderabad| 29|     Savings| 2023-01-10|             1203|
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|             1170|
|          3| Arjun|   Mumbai| 27|     Savings| 2023-03-14|             1140|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|             1108|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|             1082|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|             1052|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|             1040|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|             1022|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|             1017|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|     

In [90]:
# 85
transactions.withColumn("days_since_txn", datediff(current_date(), col("txn_date"))).show()


+------+----------+--------+------+----------+--------------+
|txn_id|account_id|txn_type|amount|  txn_date|days_since_txn|
+------+----------+--------+------+----------+--------------+
|     1|      1001|  Credit| 25000|2024-03-01|           787|
|     2|      1002|   Debit| 15000|2024-03-01|           787|
|     3|      1003|  Credit| 10000|2024-03-02|           786|
|     4|      1004|   Debit|  5000|2024-03-02|           786|
|     5|      1005|  Credit| 30000|2024-03-03|           785|
|     6|      1006|   Debit| 20000|2024-03-03|           785|
|     7|      1007|  Credit|  8000|2024-03-04|           784|
|     8|      1008|   Debit| 12000|2024-03-04|           784|
|     9|      1009|  Credit| 40000|2024-03-05|           783|
|    10|      1010|   Debit| 35000|2024-03-05|           783|
+------+----------+--------+------+----------+--------------+



# **Section 10 — Window Functions**

In [91]:
# 86
w_city = Window.partitionBy("city").orderBy(col("balance").desc())
customers.join(accounts,"customer_id")\
.withColumn("rank", rank().over(w_city)).show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance|rank|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+----+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|   1|
|          9| Divya|Bangalore| 28|     Savings| 2023-07-15|      1009|Bangalore Central| 110000|   2|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|   1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|   1|
|          8|  Neha|    Delhi| 26|     Savings| 2023-07-10|      1008|      Delhi North|  70000|   2|
|         12|Simran|    Delhi| 25|     Savings| 2023-08-21|      1012|      Delhi North|  40000|   3|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad

In [92]:
# 87
customers.join(accounts,"customer_id")\
.withColumn("rn", row_number().over(w_city)).filter(col("rn")==1).show()

+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+---+
|customer_id|  name|     city|age|account_type|signup_date|account_id|           branch|balance| rn|
+-----------+------+---------+---+------------+-----------+----------+-----------------+-------+---+
|          2| Sneha|Bangalore| 32|     Current| 2023-02-12|      1002|Bangalore Central| 120000|  1|
|          5| Karan|  Chennai| 30|     Current| 2023-05-11|      1005|    Chennai South|  60000|  1|
|          4| Priya|    Delhi| 35|     Savings| 2023-04-15|      1004|      Delhi North|  95000|  1|
|          6| Meera|Hyderabad| 31|     Savings| 2023-06-10|      1006|   Hyderabad Main| 150000|  1|
|         10|Vikram|   Mumbai| 42|     Current| 2023-08-01|      1010|      Mumbai West| 200000|  1|
|          7|  Amit|     Pune| 38|     Current| 2023-06-22|      1007|        Pune East|  30000|  1|
+-----------+------+---------+---+------------+-----------+----------+-----------------+---

In [93]:
# 88
w_txn = Window.partitionBy("txn_type").orderBy(col("amount").desc())
transactions.withColumn("dr", dense_rank().over(w_txn)).show()

+------+----------+--------+------+----------+---+
|txn_id|account_id|txn_type|amount|  txn_date| dr|
+------+----------+--------+------+----------+---+
|     9|      1009|  Credit| 40000|2024-03-05|  1|
|     5|      1005|  Credit| 30000|2024-03-03|  2|
|     1|      1001|  Credit| 25000|2024-03-01|  3|
|     3|      1003|  Credit| 10000|2024-03-02|  4|
|     7|      1007|  Credit|  8000|2024-03-04|  5|
|    10|      1010|   Debit| 35000|2024-03-05|  1|
|     6|      1006|   Debit| 20000|2024-03-03|  2|
|     2|      1002|   Debit| 15000|2024-03-01|  3|
|     8|      1008|   Debit| 12000|2024-03-04|  4|
|     4|      1004|   Debit|  5000|2024-03-02|  5|
+------+----------+--------+------+----------+---+



In [95]:
# 89
w_acc = Window.partitionBy("account_id").orderBy(col("amount").desc())
transactions.withColumn("rn", row_number().over(w_acc)).filter(col("rn")<=2).show()

+------+----------+--------+------+----------+---+
|txn_id|account_id|txn_type|amount|  txn_date| rn|
+------+----------+--------+------+----------+---+
|     1|      1001|  Credit| 25000|2024-03-01|  1|
|     2|      1002|   Debit| 15000|2024-03-01|  1|
|     3|      1003|  Credit| 10000|2024-03-02|  1|
|     4|      1004|   Debit|  5000|2024-03-02|  1|
|     5|      1005|  Credit| 30000|2024-03-03|  1|
|     6|      1006|   Debit| 20000|2024-03-03|  1|
|     7|      1007|  Credit|  8000|2024-03-04|  1|
|     8|      1008|   Debit| 12000|2024-03-04|  1|
|     9|      1009|  Credit| 40000|2024-03-05|  1|
|    10|      1010|   Debit| 35000|2024-03-05|  1|
+------+----------+--------+------+----------+---+



In [96]:
# 90
transactions.withColumn("running_total", sum("amount").over(w_acc)).show()

+------+----------+--------+------+----------+-------------+
|txn_id|account_id|txn_type|amount|  txn_date|running_total|
+------+----------+--------+------+----------+-------------+
|     1|      1001|  Credit| 25000|2024-03-01|        25000|
|     2|      1002|   Debit| 15000|2024-03-01|        15000|
|     3|      1003|  Credit| 10000|2024-03-02|        10000|
|     4|      1004|   Debit|  5000|2024-03-02|         5000|
|     5|      1005|  Credit| 30000|2024-03-03|        30000|
|     6|      1006|   Debit| 20000|2024-03-03|        20000|
|     7|      1007|  Credit|  8000|2024-03-04|         8000|
|     8|      1008|   Debit| 12000|2024-03-04|        12000|
|     9|      1009|  Credit| 40000|2024-03-05|        40000|
|    10|      1010|   Debit| 35000|2024-03-05|        35000|
+------+----------+--------+------+----------+-------------+



In [97]:
# 91
accounts.groupBy("branch").agg(sum("balance").alias("total"))\
.withColumn("rank", rank().over(Window.orderBy(col("total").desc()))).show()


+-----------------+------+----+
|           branch| total|rank|
+-----------------+------+----+
|   Hyderabad Main|285000|   1|
|      Mumbai West|245000|   2|
|Bangalore Central|230000|   3|
|      Delhi North|205000|   4|
|    Chennai South| 60000|   5|
|        Pune East| 30000|   6|
+-----------------+------+----+



In [98]:
# 92
customers.join(accounts,"customer_id")\
.groupBy("city").agg(sum("balance").alias("total"))\
.withColumn("rank", rank().over(Window.orderBy(col("total").desc()))).show()


+---------+------+----+
|     city| total|rank|
+---------+------+----+
|Hyderabad|285000|   1|
|   Mumbai|245000|   2|
|Bangalore|230000|   3|
|    Delhi|205000|   4|
|  Chennai| 60000|   5|
|     Pune| 30000|   6|
+---------+------+----+



In [99]:
# 93
customers.join(accounts,"customer_id").join(transactions,"account_id")\
.groupBy("name").agg(sum("amount").alias("total"))\
.withColumn("rank", rank().over(Window.orderBy(col("total").desc()))).show()


+------+-----+----+
|  name|total|rank|
+------+-----+----+
| Divya|40000|   1|
|Vikram|35000|   2|
| Karan|30000|   3|
| Rahul|25000|   4|
| Meera|20000|   5|
| Sneha|15000|   6|
|  Neha|12000|   7|
| Arjun|10000|   8|
|  Amit| 8000|   9|
| Priya| 5000|  10|
+------+-----+----+



In [100]:
# 94
customers.join(accounts,"customer_id").join(transactions,"account_id")\
.groupBy("city","name").agg(sum("amount").alias("total"))\
.withColumn("rank", rank().over(Window.partitionBy("city").orderBy(col("total").desc())))\
.filter(col("rank")==1).show()

+---------+------+-----+----+
|     city|  name|total|rank|
+---------+------+-----+----+
|Bangalore| Divya|40000|   1|
|  Chennai| Karan|30000|   1|
|    Delhi|  Neha|12000|   1|
|Hyderabad| Rahul|25000|   1|
|   Mumbai|Vikram|35000|   1|
|     Pune|  Amit| 8000|   1|
+---------+------+-----+----+



In [101]:
# 95
customers.join(accounts,"customer_id").join(branches,"branch")\
.groupBy("region","name").agg(max("balance").alias("bal"))\
.withColumn("rank", rank().over(Window.partitionBy("region").orderBy(col("bal").desc())))\
.filter(col("rank")==1).show()


+------+------+------+----+
|region|  name|   bal|rank|
+------+------+------+----+
| North| Priya| 95000|   1|
| South| Meera|150000|   1|
|  West|Vikram|200000|   1|
+------+------+------+----+



# **Section 11 — Complex JSON**

In [110]:
# 96
profiles.show()

+--------------------+-----------+------+--------------------+
|             contact|customer_id|  name|            services|
+--------------------+-----------+------+--------------------+
|{rahul@mail.com, ...|          1| Rahul|[UPI, Credit Card...|
|{sneha@mail.com, ...|          2| Sneha|   [UPI, Debit Card]|
|{arjun@mail.com, ...|          3| Arjun| [Net Banking, Loan]|
|{meera@mail.com, ...|          6| Meera|[UPI, Credit Card...|
|{vikram@mail.com,...|         10|Vikram|[Net Banking, Wea...|
+--------------------+-----------+------+--------------------+



In [114]:
# 97
profiles.select("customer_id","name","contact.email","contact.phone").show()


+-----------+------+---------------+----------+
|customer_id|  name|          email|     phone|
+-----------+------+---------------+----------+
|          1| Rahul| rahul@mail.com|9000011111|
|          2| Sneha| sneha@mail.com|9000022222|
|          3| Arjun| arjun@mail.com|9000033333|
|          6| Meera| meera@mail.com|9000066666|
|         10|Vikram|vikram@mail.com|9000101010|
+-----------+------+---------------+----------+



In [112]:
# 98
profiles.select("customer_id","name",explode("services")).show()

+-----------+------+-----------+
|customer_id|  name|        col|
+-----------+------+-----------+
|          1| Rahul|        UPI|
|          1| Rahul|Credit Card|
|          1| Rahul|Net Banking|
|          2| Sneha|        UPI|
|          2| Sneha| Debit Card|
|          3| Arjun|Net Banking|
|          3| Arjun|       Loan|
|          6| Meera|        UPI|
|          6| Meera|Credit Card|
|          6| Meera|       Loan|
|         10|Vikram|Net Banking|
|         10|Vikram|     Wealth|
+-----------+------+-----------+



In [113]:
# 99
profiles.select("name",explode("services").alias("service"))\
.filter(col("service")=="UPI").show()

+-----+-------+
| name|service|
+-----+-------+
|Rahul|    UPI|
|Sneha|    UPI|
|Meera|    UPI|
+-----+-------+



In [115]:

# 100
profiles.select(explode("services").alias("service"))\
.groupBy("service").count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|     Wealth|    1|
|       Loan|    2|
|Credit Card|    2|
|Net Banking|    3|
| Debit Card|    1|
|        UPI|    3|
+-----------+-----+

